# Ler base de dados

In [ ]:
import pandas as pd
import numpy as np
equipDB = pd.read_csv("EquipDB.csv", header=None, names=["ID_equipamento", "TempoAposFalha", "Cluster", "CustoDeFalha"])
ClusterDB= pd.read_csv("ClusterDB.csv", header=None, names=["ID_Cluster", "eta", "beta"])
MPDB=pd.read_csv("MPDB.csv", header=None, names=["ID_plano_risco", "Fator_risco(k)", "CustoDoPlano"])

# Unir informações do ClusterDB ao equipDB
equipDB = equipDB.merge(ClusterDB, left_on="Cluster", right_on="ID_Cluster")


# Formula de Weibull

$P_{ij}$ é a probabilidade de falha do equipmento $i$ sob o plano de manutenção $j$

In [ ]:
def Fi(t, eta, beta):
    return 1-np.exp(-(t/eta)**beta)

def P(t0, eta, beta, k, delta_t):
    return (Fi(t=(t0+k*delta_t), eta=eta, beta=beta)-Fi(t0))/(1-Fi(t0))

# Modelo Matematicos

## Modelo para $f2(*)$


## Conjuntos e Parâmetros
$I$: Conjunto de equipamentos ($|I| = 500$, $i \in I$)

$J = \{1,2,3\}$: Planos de manutenção disponíveis ($j \in J$)

$C = \{1,2,3,4\}$: Clusters de equipamentos ($c \in C$)

$t_0^i$: Idade atual do equipamento $i$ (em anos)

$\text{c}_i$: Cluster ao qual pertence o equipamento $i$

$\text{f}_i$: Custo associado à falha do equipamento $i$

$k_j$: Fator de risco do plano de manutenção $j$

$\text{m}_j$: Custo de aplicar o plano $j$ a um equipamento

$\eta_c$: Parâmetro de escala (Weibull) para o cluster $c$

$\beta_c$: Parâmetro de forma (Weibull) para o cluster $c$

$p_{i,j}$: probabilidade de falha do equipmento $i$ sob o plano de manutenção $j$

$\Delta t = 5$ anos: Horizonte de planejamento


### Varaivel de decisão
$$
x_{i,j} = 
\begin{cases} 
1, & \text{se o equipamento } i \text{ for alocado ao plano } j \\
0, & \text{caso contrário}
\end{cases} $$

$$
\begin{align}
 f_1(\mathbf{x}) &=\min \sum_{i \in I} \sum_{j \in J} \text{m}_j \cdot x_{i,j} \quad \text{(Custo total de manutenção)} \\
f_2(\mathbf{x}) &= \min \sum_{i \in I} \sum_{j \in J} p_{i,j} \cdot \text{f}_i \cdot x_{i,j} \quad \text{(Custo esperado de falha)}
\end{align}$$
onde: 

$$
\begin{align}
p_{i,j} = \frac{F_{c(i)}(t_0^i + k_j \Delta t) - F_{c(i)}(t_0^i)}{1 - F_{c(i)}(t_0^i)} \\
F_c(t) = 1 - \exp\left[-\left(\frac{t}{\eta_c}\right)^{\beta_c}\right]
\end{align}$$

## Restrições

Alocação obrigatória
$$
\begin{equation}
\sum_{j \in J} x_{i,j} = 1 \quad \forall i \in I
\end{equation}
$$

Integralidade:

$$
\begin{equation}
x_{i,j} \in \{0,1\} \quad \forall i \in I, \forall j \in J
\end{equation}
$$

## Variavel de decisão

In [31]:
# Criar a matriz  de probabilidade Pij
Pij = np.zeros((len(equipDB), len(MPDB)))
delta_t = 5  # Definir um valor para Delta_t,

for i, equipamento in equipDB.iterrows():
    t0 = equipamento["TempoAposFalha"]
    eta = equipamento["eta"]
    beta = equipamento["beta"]
    
    for j, plano in MPDB.iterrows():
        k = plano["Fator_risco(k)"]
        Pij[i, j] = P(t0, eta, beta, k, delta_t)

# Converter a matriz para um DataFrame para melhor visualização
Pij_df = pd.DataFrame(Pij, index=equipDB["ID_equipamento"], columns=MPDB["ID_plano_risco"])


## Geração Solução Inicial (Cálculo do Índice Crítico)
Para cada equipamento $i$, calcule o índice de criticidade:

$$
\text{Índice}_i = \text{custo\_falha}_i \times p_{i,\text{nenhuma}} \times t_0^i
$$

onde:

$\text{custo\_falha}_i$: Custo associado à falha do equipamento $i$

$p_{i,\text{nenhuma}}$: Probabilidade de falha se nenhuma manutenção for aplicada (plano $j=1$)

$t_0^i$: Tempo de operação

## Alocação de Planos de Manutenção

Ordene os equipamentos em ordem $\textbf{decrescente}$ de $\text{Índice}_i$ e aloque os planos:


$\textbf{Top 20\%}$ equipamentos: Manutenção detalhada (plano $j=3$)


$\textbf{Próximos 30\%}$: Manutenção intermediária (plano $j=2$)

$\textbf{Restante 50\%}$: Nenhuma manutenção (plano $j=1$)

In [ ]:
def gerar_Xij_inicial(equipDB, MPDB, Pij):
    """Gera uma solução inicial balanceada para Xij."""
    n_equip = len(equipDB)
    Xij = np.zeros((n_equip, len(MPDB)))
    
    # Critério de criticidade: custo_falha * probabilidade sem manutenção
    criticidade = equipDB["CustoDeFalha"].values * Pij[:, 0]*equipDB["TempoAposFalha"]
    equip_ordenados = np.argsort(-criticidade)  # Ordem decrescente
    
    # Distribuição 20% detalhada, 30% intermediária, 50% nenhuma
    for idx, i in enumerate(equip_ordenados):
        if idx < 0.2 * n_equip:
            Xij[i, 2] = 1  # Plano 3 (detalhada)
        elif idx < 0.5 * n_equip:
            Xij[i, 1] = 1  # Plano 2 (intermediária)
        else:
            Xij[i, 0] = 1  # Plano 1 (nenhuma)
    
    return Xij
Xij=gerar_Xij_inicial(equipDB=equipDB, MPDB=MPDB, Pij=Pij)
Xij

array([[0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 0.],
       ...,
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.]], shape=(500, 3))